# Hafta 5 — Lojistik Regresyon ve Sınıflandırma Metrikleri

> Hücreleri **yukarıdan aşağıya sırayla** çalıştırın; sonraki hücreler önceki hücrelerde tanımlanan değişken ve fonksiyonları kullanır. Tek başına çalışmayan gösterim amaçlı kod parçaları markdown olarak bırakılmıştır. Veri setleri ilk çalıştırmada internetten indirilir.

In [ ]:
# Gerekli paketler kurulu değilse aşağıdaki satırın başındaki # işaretini kaldırıp çalıştırın
# %pip install numpy pandas matplotlib seaborn scikit-learn scipy imbalanced-learn

## Basit Anlatım: Bu Hafta Ne Öğreneceğiz?

Bazen tahmin edilmek istenen şey bir sayı değil, bir karardır: "Bu e-posta istenmeyen bir ileti mi?" ya da "Bu hasta riskli mi?" Model böyle sorulara bir olasılıkla cevap verir, örneğin "Bu e-postanın spam olma ihtimali %90." Olasılığın hangi değerden sonra "evet" sayılacağına ise biz karar veririz.

Başarıyı ölçerken dikkatli olmak gerekir. Bir grupta kişilerin yalnızca %1'i hasta olsun. Herkese "sağlıklı" diyen bir model bu grupta %99 oranında doğru görünür, ama tek bir hastayı bile tespit edemez. Bu nedenle iki soruya ayrıca bakılır: Gerçek hastaların ne kadarı yakalandı? "Hasta" denilen kişilerin ne kadarı gerçekten hasta?

---

## Bu Haftanın Hedefleri

- Lojistik regresyonun olasılık üretme mantığını anlamak
- Modeli sıfırdan kodlamak
- Karmaşıklık matrisi, precision, recall, F1 ve ROC-AUC'yi hesaplamak
- Dengesiz verilerde doğru metriği seçmek

---

## 1. Lojistik Regresyon

### Sigmoid fonksiyonu

Doğrusal regresyon −∞ ile +∞ arasında her değeri üretebilir; olasılık ise 0 ile 1 arasında olmalıdır. **Sigmoid** fonksiyonu her sayıyı bu aralığa sıkıştırır:

```
σ(z) = 1 / (1 + e⁻ᶻ)
```

| z | −4 | −2 | 0 | 2 | 4 |
|---|---|---|---|---|---|
| σ(z) | 0.02 | 0.12 | 0.50 | 0.88 | 0.98 |

### Model

```
1) Skor:      z = w·x + b
2) Olasılık:  p = σ(z)
3) Karar:     p ≥ 0.5 ise sınıf 1, değilse sınıf 0
```

**Örnek:** Çalışma saatine göre sınavı geçme. Model w = 1.5 ve b = −6 öğrenmiş olsun:

| Çalışma saati | z = 1.5x − 6 | p | Tahmin |
|---|---|---|---|
| 2 | −3.0 | 0.05 | Kaldı |
| 4 | 0.0 | 0.50 | Sınırda |
| 6 | 3.0 | 0.95 | Geçti |

p = 0.5 olduğu yer (z = 0) **karar sınırıdır**. Bu sınır düz bir çizgi olduğu için lojistik regresyon doğrusal bir sınıflandırıcıdır.

### Kayıp fonksiyonu: Çapraz entropi

Gerçek sınıf 1 iken model p = 0.9 derse küçük, p = 0.1 derse büyük ceza vermek isteriz:

```
Kayıp = −[ y·ln(p) + (1 − y)·ln(1 − p) ]
```

| Gerçek y = 1 için tahmin p | 0.99 | 0.9 | 0.5 | 0.1 |
|---|---|---|---|---|
| Kayıp | 0.01 | 0.11 | 0.69 | 2.30 |

Emin ama yanlış tahminler çok sert cezalandırılır. Bu kaybın gradyanı, doğrusal regresyondakiyle aynı sade biçimdedir: `(p − y)·x`. Model gradyan inişle eğitilir.

---

## 2. Sıfırdan Lojistik Regresyon

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

class LojistikRegresyon:
    def __init__(self, alpha=0.1, epoch=2000):
        self.alpha, self.epoch = alpha, epoch

    def fit(self, X, y):
        self.w, self.b = np.zeros(X.shape[1]), 0.0
        for _ in range(self.epoch):
            p = sigmoid(X @ self.w + self.b)
            self.w -= self.alpha * X.T @ (p - y) / len(y)   # gradyan: (p − y)·x
            self.b -= self.alpha * np.mean(p - y)
        return self

    def predict_proba(self, X):
        return sigmoid(X @ self.w + self.b)

    def predict(self, X, esik=0.5):
        return (self.predict_proba(X) >= esik).astype(int)

İki öznitelikli bir veride deneyip karar sınırını çizelim:

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                           n_clusters_per_class=1, class_sep=1.2, random_state=1)
model = LojistikRegresyon(alpha=0.5, epoch=1000).fit(X, y)
print("Doğruluk:", (model.predict(X) == y).mean())

xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
                     np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300))
P = model.predict_proba(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
plt.contourf(xx, yy, P, levels=20, cmap="RdBu_r", alpha=0.6)
plt.contour(xx, yy, P, levels=[0.5], colors="k")           # karar sınırı
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu_r", edgecolors="k"); plt.show()

scikit-learn karşılığı `LogisticRegression` sınıfıdır. Çok sınıflı problemlerde **softmax** kullanır: Her sınıf için bir skor hesaplanır ve skorlar toplamı 1 olan olasılıklara dönüştürülür.

---

## 3. Sınıflandırma Metrikleri

### Karmaşıklık matrisi

100 kişilik bir tarama testi (50 hasta, 50 sağlıklı):

| | Tahmin: Hasta | Tahmin: Sağlıklı |
|---|---|---|
| **Gerçek: Hasta** | 40 (TP — doğru yakalandı) | 10 (FN — kaçırıldı) |
| **Gerçek: Sağlıklı** | 5 (FP — yanlış alarm) | 45 (TN — doğru) |

| Metrik | Formül | Sonuç | Sorduğu soru |
|---|---|---|---|
| **Accuracy** | (TP+TN) / toplam | 85/100 = **0.85** | Tahminlerin ne kadarı doğru? |
| **Precision** | TP / (TP+FP) | 40/45 = **0.89** | "Hasta" dediklerimizin ne kadarı gerçekten hasta? |
| **Recall** | TP / (TP+FN) | 40/50 = **0.80** | Gerçek hastaların ne kadarını yakaladık? |
| **F1** | 2·P·R / (P+R) | **0.84** | Precision ve recall'un dengesi |

**Hangisi önemli?** Kanser taramasında hastayı kaçırmak (FN) çok tehlikelidir → **recall** öncelikli. Spam filtresinde önemli bir e-postayı çöpe atmak (FP) kötüdür → **precision** öncelikli.

### Eşik değeri

0.5 eşiği bir zorunluluk değildir. Eşik düşürülürse daha çok örneğe "pozitif" denir: recall artar, precision düşer. Eşik yükseltilirse tersi olur.

### ROC eğrisi ve AUC

ROC eğrisi, **tüm eşik değerleri** için iki oranı çizer: yakalanan pozitiflerin oranı (TPR) ve yanlış alarm oranı (FPR).

**Elle örnek:** Altı örnek, olasılığa göre sıralı:

| Örnek | A | B | C | D | E | F |
|---|---|---|---|---|---|---|
| Olasılık | 0.9 | 0.8 | 0.7 | 0.6 | 0.4 | 0.2 |
| Gerçek | 1 | 1 | 0 | 1 | 0 | 0 |

Eşik yukarıdan aşağı indirildikçe noktalar (FPR, TPR): (0, 0.33) → (0, 0.67) → (0.33, 0.67) → (0.33, 1) → (0.67, 1) → (1, 1)

Eğrinin altındaki alan **AUC = 0.89**'dur. AUC'nin anlamı şudur: Rastgele seçilen bir pozitif örneğe, rastgele seçilen bir negatif örnekten daha yüksek olasılık verilme ihtimali. Burada 9 (pozitif, negatif) çiftinden 8'i doğru sıralanmış: 8/9 = 0.89. AUC = 0.5 yazı-tura kadar iyi, 1.0 ise mükemmeldir.

In [ ]:
from sklearn.metrics import roc_auc_score
print(roc_auc_score([1, 1, 0, 1, 0, 0], [0.9, 0.8, 0.7, 0.6, 0.4, 0.2]))   # 0.889

### Dengesiz veri

İşlemlerin %99'unun normal olduğu bir veride her şeye "normal" diyen model %99 doğruluk elde eder ama hiçbir sahtekârlığı yakalayamaz. Dengesiz verilerde accuracy yerine precision, recall, F1 ve AUC kullanılır. Ayrıca şu yöntemler uygulanır:

- `class_weight="balanced"`: Azınlık sınıfındaki hatalara daha fazla ceza verilir.
- Eşiği iş hedefine göre ayarlamak.
- Azınlık sınıfını çoğaltmak (ör. SMOTE). Bu işlem **yalnızca eğitim verisine** uygulanır.

---

## 4. Laboratuvar: Meme Kanseri Teşhisi ve Eşik Analizi

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, precision_score, recall_score

veri = load_breast_cancer()
X, y = veri.data, 1 - veri.target        # 1 = kötü huylu (pozitif sınıf)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
sc = StandardScaler().fit(X_tr)
X_tr, X_te = sc.transform(X_tr), sc.transform(X_te)

bizim = LojistikRegresyon(alpha=0.1, epoch=3000).fit(X_tr, y_tr)
sk = LogisticRegression(max_iter=5000).fit(X_tr, y_tr)
print(classification_report(y_te, sk.predict(X_te), digits=3))
print("Bizim model doğruluk:", (bizim.predict(X_te) == y_te).mean())
ConfusionMatrixDisplay.from_estimator(sk, X_te, y_te); plt.show()

# Eşik analizi
p = sk.predict_proba(X_te)[:, 1]
print(pd.DataFrame([{"eşik": e,
                     "precision": precision_score(y_te, p >= e),
                     "recall": recall_score(y_te, p >= e)} for e in [0.2, 0.4, 0.5, 0.6, 0.8]]).round(3))

**Görevler:**
1. Eşik düştükçe precision ve recall nasıl değişiyor? Kanser taraması için hangi eşiği seçerdiniz?
2. Bir kötü huylu tümörü kaçırmanın maliyeti, gereksiz biyopsinin 10 katı olsun. `10·FN + FP` maliyetini en küçük yapan eşiği bulun.

---

## 5. Özet

- Lojistik regresyon `p = σ(wx + b)` ile olasılık üretir; karar sınırı doğrusaldır.
- Kayıp fonksiyonu çapraz entropidir; emin ama yanlış tahminleri sert cezalandırır.
- **Precision:** "pozitif dediklerimin ne kadarı doğru?" **Recall:** "pozitiflerin ne kadarını buldum?"
- **AUC**, modelin eşikten bağımsız sıralama başarısını ölçer.
- Dengesiz verilerde accuracy yanıltıcıdır.

## Kendinizi Sınayın

1. z = −1 için σ(z) yaklaşık kaçtır? Model hangi sınıfı tahmin eder?
2. TP = 30, FP = 20, FN = 10, TN = 940 için accuracy, precision ve recall'u hesaplayın. Accuracy neden yanıltıcı?
3. Eşiği 0.5'ten 0.3'e indirirseniz precision ve recall nasıl değişir?

## Çalışma Alanı

Alıştırmalar ve "Kendinizi Sınayın" soruları için aşağıdaki hücreleri kullanın.